In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    sum as spark_sum,
    when
)

gold_schema = "workspace.zomato_gold"

# ---------------------------------------------------------
# Expected Gold tables, keys and row counts
# ---------------------------------------------------------
gold_tables = {
    "dim_foods": {
        "key": "food_id",
        "expected_rows": 371563
    },
    "dim_users": {
        "key": "user_id",
        "expected_rows": 100000
    },
    "dim_restaurants": {
        "key": "restaurant_id",
        "expected_rows": 148541
    },
    "daily_sales": {
        "key": "order_date",
        "expected_rows": 209
    },
    "restaurant_performance": {
        "key": "restaurant_id",
        "expected_rows": 281
    },
    "customer_order_analysis": {
        "key": "user_id",
        "expected_rows": 281
    },
    "food_menu_analysis": {
        "key": "menu_id",
        "expected_rows": 665517
    }
}

validation_results = []

# ---------------------------------------------------------
# Validate each Gold table
# ---------------------------------------------------------
for table_name, config in gold_tables.items():

    table_full_name = f"{gold_schema}.{table_name}"

    try:
        df = spark.table(table_full_name)

        row_count = df.count()
        column_count = len(df.columns)

        key = config["key"]

        null_key_count = (
            df.filter(col(key).isNull()).count()
        )

        distinct_key_count = (
            df.select(key)
              .distinct()
              .count()
        )

        duplicate_key_count = (
            row_count - distinct_key_count
        )

        expected_rows = config["expected_rows"]

        row_count_status = (
            "PASS"
            if row_count == expected_rows
            else "FAIL"
        )

        null_key_status = (
            "PASS"
            if null_key_count == 0
            else "FAIL"
        )

        uniqueness_status = (
            "PASS"
            if duplicate_key_count == 0
            else "FAIL"
        )

        overall_status = (
            "PASS"
            if (
                row_count_status == "PASS"
                and null_key_status == "PASS"
                and uniqueness_status == "PASS"
            )
            else "FAIL"
        )

        validation_results.append(
            (
                table_name,
                row_count,
                expected_rows,
                column_count,
                null_key_count,
                distinct_key_count,
                duplicate_key_count,
                row_count_status,
                null_key_status,
                uniqueness_status,
                overall_status
            )
        )

    except Exception as e:

        validation_results.append(
            (
                table_name,
                None,
                config["expected_rows"],
                None,
                None,
                None,
                None,
                "FAIL",
                "FAIL",
                "FAIL",
                "FAIL"
            )
        )

# ---------------------------------------------------------
# Create validation report
# ---------------------------------------------------------
validation_df = spark.createDataFrame(
    validation_results,
    [
        "table_name",
        "actual_rows",
        "expected_rows",
        "column_count",
        "null_key_count",
        "distinct_key_count",
        "duplicate_key_count",
        "row_count_check",
        "null_key_check",
        "uniqueness_check",
        "overall_status"
    ]
)

display(
    validation_df.orderBy("table_name")
)

# ---------------------------------------------------------
# Validate known orphan records in food_menu_analysis
# ---------------------------------------------------------
food_menu_df = spark.table(
    f"{gold_schema}.food_menu_analysis"
)

unmatched_restaurant_count = (
    food_menu_df
    .filter(
        col("restaurant_id").isNotNull()
        & col("restaurant_name").isNull()
    )
    .count()
)

unmatched_food_count = (
    food_menu_df
    .filter(col("food_id").isNull())
    .count()
)

print("==============================================")
print("GOLD LAYER FINAL QUALITY VALIDATION")
print("==============================================")

print(f"Gold tables validated       : {len(gold_tables)}")
print(f"Unmatched menu restaurants  : {unmatched_restaurant_count}")
print(f"Unmatched menu foods        : {unmatched_food_count}")

print("==============================================")

# ---------------------------------------------------------
# Final overall result
# ---------------------------------------------------------
failed_tables = (
    validation_df
    .filter(col("overall_status") == "FAIL")
    .count()
)

if (
    failed_tables == 0
    and unmatched_restaurant_count == 160
    and unmatched_food_count == 0
):
    print("FINAL GOLD LAYER STATUS: PASS")
else:
    print("FINAL GOLD LAYER STATUS: REVIEW REQUIRED")

table_name,actual_rows,expected_rows,column_count,null_key_count,distinct_key_count,duplicate_key_count,row_count_check,null_key_check,uniqueness_check,overall_status
customer_order_analysis,281,281,15,0,281,0,PASS,PASS,PASS,PASS
daily_sales,209,209,5,0,209,0,PASS,PASS,PASS,PASS
dim_foods,371563,371563,3,0,371563,0,PASS,PASS,PASS,PASS
dim_restaurants,148541,148541,11,0,148541,0,PASS,PASS,PASS,PASS
dim_users,100000,100000,10,0,100000,0,PASS,PASS,PASS,PASS
food_menu_analysis,665517,665517,9,0,624444,41073,PASS,PASS,FAIL,FAIL
restaurant_performance,281,281,11,0,281,0,PASS,PASS,PASS,PASS


GOLD LAYER FINAL QUALITY VALIDATION
Gold tables validated       : 7
Unmatched menu restaurants  : 160
Unmatched menu foods        : 0
FINAL GOLD LAYER STATUS: REVIEW REQUIRED


In [0]:
from pyspark.sql.functions import col, count

food_menu_df = spark.table(
    "workspace.zomato_gold.food_menu_analysis"
)

duplicate_menu_ids_df = (
    food_menu_df
    .groupBy("menu_id")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)

print(
    "Duplicate menu IDs:",
    duplicate_menu_ids_df.count()
)

display(
    duplicate_menu_ids_df.limit(20)
)

Duplicate menu IDs: 41073


menu_id,count
mn785222,2
mn698549,2
mn893302,2
mn47538,2
mn17440,2
mn908830,2
mn252566,2
mn585776,2
mn910318,2
mn451157,2


In [0]:
duplicate_ids = (
    duplicate_menu_ids_df
    .select("menu_id")
    .limit(10)
)

display(
    food_menu_df
    .join(
        duplicate_ids,
        "menu_id",
        "inner"
    )
    .orderBy("menu_id")
)

menu_id,restaurant_id,restaurant_name,city,food_id,food_name,food_type,cuisine,menu_price
mn1006420,553484,FISH MAGIC,"HSR,Bangalore",fd3907,Mango,Veg,"South Indian,Snacks",80.0
mn1006420,553484,FISH MAGIC,"HSR,Bangalore",fd879091,Mango,Non-veg,"South Indian,Snacks",80.0
mn1013438,8174,Corner House Ice Cream,"HSR,Bangalore",fd777971,Cake Fudge,Veg,"Ice Cream,Desserts",150.0
mn1013438,8174,Corner House Ice Cream,"HSR,Bangalore",fd885506,Cake Fudge,Non-veg,"Ice Cream,Desserts",150.0
mn127952,459178,Balaji Bhaji Pav Dosa,"Vastrapur,Ahmedabad",fd143466,Masala Pav,Non-veg,"South Indian,Street Food",120.0
mn127952,459178,Balaji Bhaji Pav Dosa,"Vastrapur,Ahmedabad",fd17772,Masala Pav,Veg,"South Indian,Street Food",120.0
mn189498,530575,Meal Avid,"Ghatlodia,Ahmedabad",fd47222,Paneer Butter Masala,Non-veg,"North Indian,Indian",230.0
mn189498,530575,Meal Avid,"Ghatlodia,Ahmedabad",fd2461,Paneer Butter Masala,Veg,"North Indian,Indian",230.0
mn650723,258392,The Lassi Day,Anantapur,fd669322,Aloo Tikki Burger,Non-veg,"Pizzas,Beverages",130.0
mn650723,258392,The Lassi Day,Anantapur,fd0,Aloo Tikki Burger,Veg,"Pizzas,Beverages",130.0


In [0]:
from pyspark.sql.functions import col, count

menus_silver_df = spark.table(
    "workspace.zomato_silver.menus"
)

print("Silver menu rows:", menus_silver_df.count())

print(
    "Distinct menu IDs:",
    menus_silver_df.select("menu_id").distinct().count()
)

print(
    "Duplicate menu IDs:",
    menus_silver_df
    .groupBy("menu_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

Silver menu rows: 665517
Distinct menu IDs: 624444
Duplicate menu IDs: 41073


In [0]:
display(
    menus_silver_df
    .groupBy("menu_id")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
    .limit(20)
)

menu_id,count
mn785222,2
mn698549,2
mn893302,2
mn47538,2
mn17440,2
mn908830,2
mn252566,2
mn585776,2
mn910318,2
mn451157,2


In [0]:
print(
    "Distinct menu_id + food_id:",
    menus_silver_df
    .select("menu_id", "f_id")
    .distinct()
    .count()
)

Distinct menu_id + food_id: 665517


In [0]:
from pyspark.sql.functions import col

gold_schema = "workspace.zomato_gold"

# ---------------------------------------------------------
# Expected Gold tables and row counts
# ---------------------------------------------------------
gold_tables = {
    "dim_foods": {
        "keys": ["food_id"],
        "expected_rows": 371563
    },
    "dim_users": {
        "keys": ["user_id"],
        "expected_rows": 100000
    },
    "dim_restaurants": {
        "keys": ["restaurant_id"],
        "expected_rows": 148541
    },
    "daily_sales": {
        "keys": ["order_date"],
        "expected_rows": 209
    },
    "restaurant_performance": {
        "keys": ["restaurant_id"],
        "expected_rows": 281
    },
    "customer_order_analysis": {
        "keys": ["user_id"],
        "expected_rows": 281
    },
    "food_menu_analysis": {
        "keys": ["menu_id", "food_id"],
        "expected_rows": 665517
    }
}

validation_results = []

# ---------------------------------------------------------
# Validate Gold tables
# ---------------------------------------------------------
for table_name, config in gold_tables.items():

    table_full_name = f"{gold_schema}.{table_name}"

    try:
        df = spark.table(table_full_name)

        row_count = df.count()
        column_count = len(df.columns)

        keys = config["keys"]

        null_key_count = (
            df.filter(
                col(keys[0]).isNull()
                if len(keys) == 1
                else (
                    col(keys[0]).isNull()
                    | col(keys[1]).isNull()
                )
            ).count()
        )

        distinct_key_count = (
            df.select(*keys)
              .distinct()
              .count()
        )

        duplicate_key_count = (
            row_count - distinct_key_count
        )

        expected_rows = config["expected_rows"]

        row_count_status = (
            "PASS"
            if row_count == expected_rows
            else "FAIL"
        )

        null_key_status = (
            "PASS"
            if null_key_count == 0
            else "FAIL"
        )

        uniqueness_status = (
            "PASS"
            if duplicate_key_count == 0
            else "FAIL"
        )

        overall_status = (
            "PASS"
            if (
                row_count_status == "PASS"
                and null_key_status == "PASS"
                and uniqueness_status == "PASS"
            )
            else "FAIL"
        )

        validation_results.append(
            (
                table_name,
                row_count,
                expected_rows,
                column_count,
                null_key_count,
                distinct_key_count,
                duplicate_key_count,
                row_count_status,
                null_key_status,
                uniqueness_status,
                overall_status
            )
        )

    except Exception as e:

        validation_results.append(
            (
                table_name,
                None,
                config["expected_rows"],
                None,
                None,
                None,
                None,
                "FAIL",
                "FAIL",
                "FAIL",
                "FAIL"
            )
        )

# ---------------------------------------------------------
# Validation report
# ---------------------------------------------------------
validation_df = spark.createDataFrame(
    validation_results,
    [
        "table_name",
        "actual_rows",
        "expected_rows",
        "column_count",
        "null_key_count",
        "distinct_key_count",
        "duplicate_key_count",
        "row_count_check",
        "null_key_check",
        "uniqueness_check",
        "overall_status"
    ]
)

display(
    validation_df.orderBy("table_name")
)

# ---------------------------------------------------------
# Known relationship checks
# ---------------------------------------------------------
food_menu_df = spark.table(
    f"{gold_schema}.food_menu_analysis"
)

unmatched_restaurant_count = (
    food_menu_df
    .filter(
        col("restaurant_id").isNotNull()
        & col("restaurant_name").isNull()
    )
    .count()
)

unmatched_food_count = (
    food_menu_df
    .filter(
        col("food_id").isNull()
    )
    .count()
)

print("==============================================")
print("GOLD LAYER FINAL QUALITY VALIDATION")
print("==============================================")

print(
    f"Gold tables validated      : {len(gold_tables)}"
)

print(
    f"Unmatched menu restaurants : {unmatched_restaurant_count}"
)

print(
    f"Unmatched menu foods       : {unmatched_food_count}"
)

print("==============================================")

# ---------------------------------------------------------
# Final status
# ---------------------------------------------------------
failed_tables = (
    validation_df
    .filter(col("overall_status") == "FAIL")
    .count()
)

if (
    failed_tables == 0
    and unmatched_restaurant_count == 160
    and unmatched_food_count == 0
):
    print("FINAL GOLD LAYER STATUS: PASS")
else:
    print("FINAL GOLD LAYER STATUS: REVIEW REQUIRED")

table_name,actual_rows,expected_rows,column_count,null_key_count,distinct_key_count,duplicate_key_count,row_count_check,null_key_check,uniqueness_check,overall_status
customer_order_analysis,281,281,15,0,281,0,PASS,PASS,PASS,PASS
daily_sales,209,209,5,0,209,0,PASS,PASS,PASS,PASS
dim_foods,371563,371563,3,0,371563,0,PASS,PASS,PASS,PASS
dim_restaurants,148541,148541,11,0,148541,0,PASS,PASS,PASS,PASS
dim_users,100000,100000,10,0,100000,0,PASS,PASS,PASS,PASS
food_menu_analysis,665517,665517,9,0,665517,0,PASS,PASS,PASS,PASS
restaurant_performance,281,281,11,0,281,0,PASS,PASS,PASS,PASS


GOLD LAYER FINAL QUALITY VALIDATION
Gold tables validated      : 7
Unmatched menu restaurants : 160
Unmatched menu foods       : 0
FINAL GOLD LAYER STATUS: PASS
